In [2]:
import pandas as pd
from pathlib import Path

API_PATH = Path("../data_app/debug/teams_from_api.csv")
HIST_PATH = Path("../data_app/debug/debug_hist_teams.csv")

api = pd.read_csv(API_PATH)
hist = pd.read_csv(HIST_PATH)

api.columns = ["team_api"]
hist.columns = ["team_hist"]

print("API teams:", len(api))
print("History teams:", len(hist))

display(api.head())
display(hist.head())


API teams: 96
History teams: 167


,team_api
0,1. FC Heidenheim 1846
1,1. FC Köln
2,1. FC Union Berlin
3,1. FSV Mainz 05
4,AC Milan


,team_hist
0,Ajaccio
1,Ajaccio GFCO
2,Alaves
3,Almeria
4,Amiens


In [3]:
api_set = set(api["team_api"])
hist_set = set(hist["team_hist"])

only_api = sorted(api_set - hist_set)
only_hist = sorted(hist_set - api_set)

print("❌ Tylko w API (brak w historii):", len(only_api))
print("❌ Tylko w historii (brak w API):", len(only_hist))

pd.DataFrame(only_api, columns=["team_api_only"]).head(20)


❌ Tylko w API (brak w historii): 94
❌ Tylko w historii (brak w API): 165


,team_api_only
0,1. FC Heidenheim 1846
1,1. FC Köln
2,1. FC Union Berlin
3,1. FSV Mainz 05
4,AC Milan
5,AC Pisa 1909
6,ACF Fiorentina
7,AFC Bournemouth
8,AJ Auxerre
9,AS Monaco FC


In [10]:
import re
import unicodedata
import pandas as pd

def strip_accents(s: str) -> str:
    s = unicodedata.normalize("NFKD", s)
    return "".join(ch for ch in s if not unicodedata.combining(ch))

def normalize_team_name(s: str) -> str:
    if pd.isna(s):
        return ""
    s = str(s).strip().lower()
    s = strip_accents(s)

    # usuń typowe "suffixy" z API
    s = re.sub(r"\b(fc|cf|afc|sc|sv|ac|1|as|ssc|ogc|rcd|rc|us|tsg)\b", " ", s)
    s = re.sub(r"\b(1909|1910|1913|1846|1901|29)\b", " ", s)

    # ujednolicenia znaków
    s = s.replace("&", "and")
    s = s.replace("'", "")
    s = s.replace(".", "")
    s = s.replace("-", " ")

    # usuń wielokrotne spacje
    s = re.sub(r"\s+", " ", s).strip()
    return s

api["norm"] = api["team_api"].apply(normalize_team_name)
hist["norm"] = hist["team_hist"].apply(normalize_team_name)

display(api.head(10))
display(hist.head(10))


,team_api,norm
0,1. FC Heidenheim 1846,heidenheim
1,1. FC Köln,koln
2,1. FC Union Berlin,union berlin
3,1. FSV Mainz 05,fsv mainz 05
4,AC Milan,milan
5,AC Pisa 1909,pisa
6,ACF Fiorentina,acf fiorentina
7,AFC Bournemouth,bournemouth
8,AJ Auxerre,aj auxerre
9,AS Monaco FC,monaco


,team_hist,norm
0,Ajaccio,ajaccio
1,Ajaccio GFCO,ajaccio gfco
2,Alaves,alaves
3,Almeria,almeria
4,Amiens,amiens
5,Angers,angers
6,Arsenal,arsenal
7,Aston Villa,aston villa
8,Atalanta,atalanta
9,Athletic Bilbao,athletic bilbao


In [11]:
auto = hist.merge(
    api,
    on="norm",
    how="left",
    suffixes=("_hist", "_api")
)[["team_hist", "team_api", "norm"]]

print("Auto dopasowane:", auto["team_api"].notna().sum(), "z", len(auto))
missing = auto[auto["team_api"].isna()].copy()

print("Brak dopasowania:", len(missing))
display(auto.head(20))
display(missing.head(50))


Auto dopasowane: 48 z 167
Brak dopasowania: 119


,team_hist,team_api,norm
0,Ajaccio,NaN,ajaccio
1,Ajaccio GFCO,NaN,ajaccio gfco
2,Alaves,NaN,alaves
3,Almeria,NaN,almeria
4,Amiens,NaN,amiens
5,Angers,NaN,angers
6,Arsenal,Arsenal FC,arsenal
7,Aston Villa,Aston Villa FC,aston villa
8,Atalanta,NaN,atalanta
9,Athletic Bilbao,NaN,athletic bilbao


,team_hist,team_api,norm
0,Ajaccio,NaN,ajaccio
1,Ajaccio GFCO,NaN,ajaccio gfco
2,Alaves,NaN,alaves
3,Almeria,NaN,almeria
4,Amiens,NaN,amiens
5,Angers,NaN,angers
8,Atalanta,NaN,atalanta
9,Athletic Bilbao,NaN,athletic bilbao
10,Atletico Madrid,NaN,atletico madrid
12,Auxerre,NaN,auxerre


In [19]:
MANUAL_MAP = {
    "Alaves": "Deportivo Alavés",
    "Angers": "Angers SCO",
    "Arsenal": "Arsenal FC",
    "Aston Villa": "Aston Villa FC",
    "Atalanta": "Atalanta BC",
    "Athletic Bilbao": "Athletic Club",
    "Atletico Madrid": "Club Atlético de Madrid",
    "Augsburg": "FC Augsburg",
    "Auxerre": "AJ Auxerre",
    "Barcelona": "FC Barcelona",
    "Bayern Munich": "FC Bayern München",
    "Bologna": "Bologna FC 1909",
    "Bournemouth": "AFC Bournemouth",
    "Brentford":"Brentford FC",
    "Brest": "Stade Brestois 29",
    "Brighton": "Brighton & Hove Albion FC",
    "Burnley": "Burnley FC",
    "Cagliari": "Cagliari Calcio",
    "Celta": "RC Celta de Vigo",
    "Chelsea": "Chelsea FC",
    "Como": "Como 1907",
    "Cremonese": "US Cremonese",
    "Crystal Palace": "Crystal Palace FC",
    "Dortmund": "Borussia Dortmund",
    "Ein Frankfurt": "Eintracht Frankfurt",
    "Elche": "Elche CF",
    "Espanol": "RCD Espanyol de Barcelona",
    "Everton": "Everton FC",
    "FC Koln": "1. FC Köln",
    "Fiorentina": "ACF Fiorentina",
    "Freiburg": "SC Freiburg",
    "Fulham": "Fulham FC",
    "Genoa": "Genoa CFC",
    "Getafe": "Getafe CF",
    "Girona": "Girona FC",
    "Hamburg": "Hamburger SV",
    "Heidenheim":"1. FC Heidenheim 1846",
    "Hoffenheim": "TSG 1899 Hoffenheim",
    "Inter Milan": "FC Internazionale Milano",
    "Juventus": "Juventus FC",
    "Lazio": "SS Lazio",
    "Le Havre": "Le Havre AC",
    "Lecce": "US Lecce",
    "Leeds": "Leeds United FC",
    "Lens": "Racing Club de Lens",
    "Levante": "Levante UD",
    "Leverkusen": "Bayer 04 Leverkusen",
    "Lille": "Lille OSC",
    "Liverpool": "Liverpool FC",
    "Lorient": "FC Lorient",
    "Lyon": "Olympique Lyonnais",
    "M'gladbach": "Borussia Mönchengladbach",
    "Mainz": "1. FSV Mainz 05",
    "Mallorca": "RCD Mallorca",
    "Manchester City": "Manchester City FC",
    "Manchester United": "Manchester United FC",
    "Marseille": "Olympique de Marseille",
    "Metz": "FC Metz",
    "Milan": "AC Milan",
    "Monaco": "AS Monaco FC",
    "Nantes": "FC Nantes",
    "Napoli": "SSC Napoli",
    "Newcastle": "Newcastle United FC",
    "Nice": "OGC Nice",
    "Nott'm Forest": "Nottingham Forest FC",
    "Osasuna": "CA Osasuna",
    "Oviedo": "Real Oviedo",
    "Paris FC": "Paris FC",
    "Paris Saint-Germain": "Paris Saint-Germain FC",
    "Parma": "Parma Calcio 1913",
    "Pisa": "AC Pisa 1909",
    "RB Leipzig": "RB Leipzig",
    "Real Betis": "Real Betis Balompié",
    "Real Madrid": "Real Madrid CF",
    "Rennes": "Stade Rennais FC 1901",
    "Roma": "AS Roma",
    "Sassuolo": "US Sassuolo Calcio",
    "Sevilla": "Sevilla FC",
    "Sociedad": "Real Sociedad de Fútbol",
    "St Pauli": "FC St. Pauli 1910",
    "Strasbourg": "RC Strasbourg Alsace",
    "Stuttgart": "VfB Stuttgart",
    "Sunderland": "Sunderland AFC",
    "Torino": "Torino FC",
    "Tottenham": "Tottenham Hotspur FC",
    "Toulouse": "Toulouse FC",
    "Udinese": "Udinese Calcio",
    "Union Berlin": "1. FC Union Berlin",
    "Valencia": "Valencia CF",
    "Vallecano": "Rayo Vallecano de Madrid",
    "Verona": "Hellas Verona FC",
    "Villarreal": "Villarreal CF",
    "Werder Bremen": "SV Werder Bremen",
    "West Ham": "West Ham United FC",
    "Wolfsburg": "VfL Wolfsburg",
    "Wolverhampton": "Wolverhampton Wanderers FC"
}


In [20]:
auto_fixed = auto.copy()

auto_fixed["team_api_final"] = (
    auto_fixed["team_api"]
    .fillna(auto_fixed["team_hist"].map(MANUAL_MAP))
)

still_missing = auto_fixed[auto_fixed["team_api_final"].isna()]

print("Po manual map:")
print("Brak dopasowania:", len(still_missing))

display(auto_fixed.head(20))
display(still_missing[["team_hist"]].head(50))


Po manual map:
Brak dopasowania: 71


,team_hist,team_api,norm,team_api_final
0,Ajaccio,NaN,ajaccio,NaN
1,Ajaccio GFCO,NaN,ajaccio gfco,NaN
2,Alaves,NaN,alaves,Deportivo Alavés
3,Almeria,NaN,almeria,NaN
4,Amiens,NaN,amiens,NaN
5,Angers,NaN,angers,Angers SCO
6,Arsenal,Arsenal FC,arsenal,Arsenal FC
7,Aston Villa,Aston Villa FC,aston villa,Aston Villa FC
8,Atalanta,NaN,atalanta,Atalanta BC
9,Athletic Bilbao,NaN,athletic bilbao,Athletic Club


,team_hist
0,Ajaccio
1,Ajaccio GFCO
3,Almeria
4,Amiens
14,Bastia
16,Benevento
17,Bielefeld
18,Bochum
20,Bordeaux
23,Brescia
